# Uncertainty Quantification

Uncertainty quantification is required to ensure that the estimates of the parameters are
close to their true values. This parameter accuracy is measured by computing the covariance matrix. The diagonal of this 
matrix contains the variance of the estimated parameters. The uncertainty in the parameters is taken as the standard 
deviation of the variance from the covariance matrix.

Assuming Gaussian independent and identically distributed measurement errors, the covariance matrix 
can be computed using three methods in ParmEst.

1. Reduced Hessian Method

2. Finite Difference Method

3. Automatic Differentiation Method

These methods are described in more details in the sections below.

## Computing the covariance matrix with ParmEst

This section describes the steps in computing the covariance matrix in ParmEst.

### Import the necessary packages

In [1]:
import sys
import pandas as pd
import pyomo.contrib.parmest.parmest as parmest

# If running on Google Colab, install Pyomo and Ipopt via IDAES
on_colab = "google.colab" in sys.modules
if on_colab:
    !wget "https://raw.githubusercontent.com/dowlinglab/pyomo-doe/main/notebooks/tclab_pyomo.py"

# import TCLab model, simulation, and data analysis functions
from tclab_pyomo import (
    TC_Lab_data,
    TC_Lab_experiment,
    extract_results,
    extract_plot_results,
)

### Load the experimental data

In [2]:
# load experimental data
if on_colab:
    file = "https://raw.githubusercontent.com/dowlinglab/pyomo-doe/main/data/tclab_sine_test_5min_period.csv"
else:
    file = '../data/tclab_sine_test_5min_period.csv'
df = pd.read_csv(file)

# store in custom data class 
tc_data = TC_Lab_data(
    name="Sine Wave Test for Heater 1",
    time=df['Time'].values,
    T1=df['T1'].values,
    u1=df['Q1'].values,
    P1=200,
    TS1_data=None,
    T2=df['T2'].values,
    u2=df['Q2'].values,
    P2=200,
    TS2_data=None,
    Tamb=df['T1'].values[0],
)

# set default number of states in the TCLab model
number_tclab_states = 2

### Get the parameter estimates

In [3]:
# define an Experiment object within parmest
TC_Lab_sine_exp = TC_Lab_experiment(data=tc_data, number_of_states=number_tclab_states)

# define an Estimator object using the weighted sum of squared errors objective function
pest = parmest.Estimator([TC_Lab_sine_exp, ], obj_function='SSE_weighted', tee=True)

# estimate the paramters
obj, theta = pest.theta_est()

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

### Covariance Matrix via Reduced Hessian
When the objective function is the sum of squared errors (SSE) for homogeneous data, defined as
$\text{SSE} = \sum_{i = 1}^{n} \left(\boldsymbol{y}_{i} - \boldsymbol{f}(\boldsymbol{x}_{i};
\boldsymbol{\theta})\right)^\text{T} \left(\boldsymbol{y}_{i} - \boldsymbol{f}(\boldsymbol{x}_{i};
\boldsymbol{\theta})\right)$, the covariance matrix can be computed as:

$$
   \boldsymbol{V}_{\boldsymbol{\theta}} = 2 \sigma^2 \left(\frac{\partial^2 \text{SSE}}
    {\partial \boldsymbol{\theta}^2}\right)^{-1}_{\boldsymbol{\theta}
    = \hat{\boldsymbol{\theta}}}
$$

Similarly, when the objective function is the weighted SSE (WSSE) for heterogeneous data, defined as
$\text{WSSE} = \frac{1}{2} \sum_{i = 1}^{n} \left(\boldsymbol{y}_{i} -
\boldsymbol{f}(\boldsymbol{x}_{i};\boldsymbol{\theta})\right)^\text{T} \boldsymbol{\Sigma}_{\boldsymbol{y}}^{-1}
\left(\boldsymbol{y}_{i} - \boldsymbol{f}(\boldsymbol{x}_{i};\boldsymbol{\theta})\right)$,
the covariance matrix is:

$$
   \boldsymbol{V}_{\boldsymbol{\theta}} = \left(\frac{\partial^2 \text{WSSE}}
    {\partial \boldsymbol{\theta}^2}\right)^{-1}_{\boldsymbol{\theta}
    = \hat{\boldsymbol{\theta}}}
$$

Where $\boldsymbol{V}_{\boldsymbol{\theta}}$ is the covariance matrix of the estimated
parameters $\hat{\boldsymbol{\theta}} \in \mathbb{R}^p$, $\boldsymbol{y}_{i} \in \mathbb{R}^m$ are
observations of the measured variables, $\boldsymbol{f}$ is the model function,
$\boldsymbol{x}_{i} \in \mathbb{R}^{q}$ are the input variables, $n$ is the number of experiments,
$\boldsymbol{\Sigma}_{\boldsymbol{y}}$ is the measurement error covariance matrix, and $\sigma^2$
is the variance of the measurement error.

This method computes the inverse of the Hessian using PyNumero by scaling the objective function (SSE or WSSE) with a constant probability factor, $\frac{1}{n}$.

In [4]:
# compute the covariance using the reduced hessian method in ParmEst
cov_red = pest.cov_est(method="reduced_hessian")

Ipopt 3.13.2: bound_relax_factor=0
honor_original_bounds=no


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-sca

### Covariance Matrix via Finite Difference
In this method, the covariance matrix, $\boldsymbol{V}_{\boldsymbol{\theta}}$, is
computed by differentiating the Hessian,
$\frac{\partial^2 \text{SSE}}{\partial \boldsymbol{\theta}^2}$
or
$\frac{\partial^2 \text{WSSE}}{\partial \boldsymbol{\theta}^2}$, and
applying Gauss-Newton approximation which results in:

$$
   \boldsymbol{V}_{\boldsymbol{\theta}} = \left(\sum_{i = 1}^n \boldsymbol{G}_{i}^{\text{T}}
    \boldsymbol{\Sigma}_{\boldsymbol{y}}^{-1} \boldsymbol{G}_{i} \right)^{-1}
$$

where

$$
   \boldsymbol{G}_{i} = \frac{\partial \boldsymbol{f}(\boldsymbol{x}_i;\boldsymbol{\theta})}
    {\partial \boldsymbol{\theta}} \approx \frac{\boldsymbol{f}(\boldsymbol{x}_i;\theta_k + \Delta \theta_k)
    \vert_{\hat{\boldsymbol{\theta}}} - \boldsymbol{f}(\boldsymbol{x}_i;\theta_k - \Delta \theta_k)
    \vert_{\hat{\boldsymbol{\theta}}}}{2 \Delta \theta_k} \quad \forall \quad \theta_k \, \in \,
    [\theta_1,\cdots, \theta_p]
$$

In [5]:
# compute the covariance using the finite difference method in ParmEst
cov_finite = pest.cov_est(method="finite_difference")

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

### Covariance Matrix via Automatic Differentiation
Similar to the finite difference method, the covariance matrix can be computed as:

$$
   \boldsymbol{V}_{\boldsymbol{\theta}} = \left( \sum_{i = 1}^n \boldsymbol{G}_{\text{kaug},\, i}^{\text{T}}
    \boldsymbol{\Sigma}_{\boldsymbol{y}}^{-1} \boldsymbol{G}_{\text{kaug},\, i} \right)^{-1}
$$

However, this method uses implicit differentiation and the model-optimality or Karush–Kuhn–Tucker (KKT) conditions
to compute the Jacobian matrix, $\boldsymbol{G}_{\text{kaug},\, i}$, for experiment $i$.

$$
   \boldsymbol{G}_{\text{kaug},\,i} = \frac{\partial \boldsymbol{f}(\boldsymbol{x}_i,\boldsymbol{\theta})}
    {\partial \boldsymbol{\theta}} + \frac{\partial \boldsymbol{f}(\boldsymbol{x}_i,\boldsymbol{\theta})}
    {\partial \boldsymbol{x}_i}\frac{\partial \boldsymbol{x}_i}{\partial \boldsymbol{\theta}}
$$

In [6]:
cov_auto = pest.cov_est(method="automatic_differentiation_kaug")

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

### Compare the covariance matrix results

In [7]:
print("The covariance matrix from the Reduced Hessian method is:\n", cov_red)

print("\nThe covariance matrix from the Finite Difference method is:\n", cov_finite)

print("\nThe covariance matrix from the Automatic Differentiation method is:\n", cov_auto)

The covariance matrix from the Reduced Hessian method is:
                    Ua            Ub       inv_CpH       inv_CpS
Ua       1.866396e-10 -1.189291e-11  1.295112e-09 -5.182846e-07
Ub      -1.194517e-11  1.537956e+02  8.051574e+01 -5.798823e+06
inv_CpH  1.295105e-09  8.051574e+01  4.215194e+01 -3.035824e+06
inv_CpS -5.107547e-07 -5.798823e+06 -3.035824e+06  2.186431e+11

The covariance matrix from the Finite Difference method is:
                    Ua            Ub       inv_CpH       inv_CpS
Ua       2.028463e-10  9.211540e-04  4.822480e-04 -3.473380e+01
Ub       9.211540e-04  4.949254e+04  2.591056e+04 -1.866207e+09
inv_CpH  4.822480e-04  2.591056e+04  1.356481e+04 -9.770050e+08
inv_CpS -3.473380e+01 -1.866207e+09 -9.770050e+08  7.036874e+13

The covariance matrix from the Automatic Differentiation method is:
                    Ua            Ub       inv_CpH       inv_CpS
Ua       1.857019e-10  1.318805e-07  7.032935e-08 -4.973750e-03
Ub       1.318914e-07  3.299503e+04  1.72